# AUDIT-X FINAL · E · Deduplicated evaluation (CPU) — sensitivity analysis

The final runs kept every duplicate group inside one partition, but four MaleVis clusters of identical-fingerprint images are larger
than the 250-image cap (VBA 500, Vilsel 294, Hlux 264, Amonetize 260); the cap cut them into pieces, and in four of five splits a piece
landed on each side of the train/test boundary. This notebook repeats **exactly the same analysis as FINAL_D** after removing every
calibration, test and unknown image that has a training image within 8 bits of its 256-bit hash (or an identical file or pixel content).
The models are unchanged. This analysis is a **deviation from PROTOCOL.md**, reported as such; FINAL_D remains the pre-registered result.

**Inputs:** identical to FINAL_D (outputs of FINAL_A2, FINAL_B, FINAL_C and of notebooks 08, 10, 14). Accelerator **None**. ≈ 2–4 h.
Results: `/kaggle/working/auditx_final_analysis_dedup/` (includes `T_dedup_removed.csv`: images removed per dataset, split and part).

**MaleVis run:** attach **FINAL_A2** (Amendment 1, primary). To repeat the analysis on the original MaleVis run as a sensitivity analysis, attach FINAL_A instead and set `MALEVIS = "v4"` in the run cell.

In [ ]:
%%writefile /kaggle/working/auditx_eval_v2.py
# =============================================================================
# AUDIT-X  —  corrected evaluation pipeline (v2)
# =============================================================================
# One script that produces every number the paper needs, from saved artefacts.
#
# What it fixes relative to the original notebooks
#   1. Known data are split per run into  train / val / calib / test  (70/10/10/10).
#      - val   : epoch selection only (known classes only)
#      - calib : operating thresholds only (target FAR)
#      - test  : FAR and closed-set accuracy are REPORTED here, so FAR is a
#                measurement, not a tautology. Training samples are never scored.
#   2. Repeated over several random unknown-class splits (SPLIT_SEEDS); results are
#      mean ± std across splits, with bootstrap CIs and paired tests across splits.
#   3. Scores are computed in float64 from logits; non-saturating scores
#      (MaxLogit, Energy) are reported next to MSP / entropy, plus the fraction of
#      calibration scores tied at the threshold (saturation diagnostic).
#   4. Threshold-free metrics (AUROC, AUPR-out, FPR@95TPR) are primary;
#      ZDR at a calibrated 5% FAR is the operational metric.
#   5. Real baselines: single CNNs, a jointly-trained average ensemble, an
#      independently-trained deep ensemble, a genuine OpenMax (Weibull on distances
#      to class MAVs), Energy, MaxLogit, KNN on penultimate features.
#   6. Four-tag triage is reported with per-tag fractions on test-known and unknown.
#   7. Optional dynamic early exit that is actually conditional: the exit rule uses
#      MobileNet's own confidence only (the gate needs all three experts, so a
#      gate-entropy exit rule can never save compute). Cost is measured with
#      torch.utils.flop_counter and wall-clock timing per expert.
#   8. Classes named in NEVER_UNKNOWN (benign / "Other") are never held out as
#      zero-day, so flagging benign files cannot count as zero-day recall.
#   9. Everything is resumable: per-run scores are saved to .npz and skipped if present.
#
# Adversarial, federated, concept-drift and SOC-stream experiments are NOT here on
# purpose; see the accompanying notes for why they need redesign first.
#
# Usage (Kaggle cell):   !python auditx_eval_v2.py
# Usage (subset):        !python auditx_eval_v2.py --datasets Malimg --seeds 0 1
# Smoke test:            python auditx_eval_v2.py --smoke
# =============================================================================
import argparse, copy, json, math, os, time, warnings, zlib
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from scipy import stats
from scipy.special import logsumexp, softmax
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)

# -----------------------------------------------------------------------------
# 0. Configuration (edit paths here, or override from the command line)
# -----------------------------------------------------------------------------
CFG = dict(
    DATASETS=OrderedDict(
        Malimg="/kaggle/input/datasets/mohammadmehedimunna/malimg-dataset-25/Malimg_Dataset_25classes",
        MaleVis="/kaggle/input/datasets/mohammadmehedimunna/melevis/MaleVis",
        Dumpware10="/kaggle/working/dumpware10_merged",   # built from dumpware10_images/224/224/{TRAINING,TEST}
    ),
    # matched case-insensitively against class folder names; these are never held out as zero-day
    NEVER_UNKNOWN=["other", "others", "benign", "goodware", "clean", "legitimate"],
    UNKNOWN_RATIO=0.2,
    SPLIT_SEEDS=[0, 1, 2, 3, 4],
    FRACS=(0.70, 0.10, 0.10, 0.10),          # train / val / calib / test (known classes)
    EPOCHS=12,
    BATCH=32,
    LR=1e-3,
    WD=1e-4,
    IMG=224,
    WORKERS=4,
    PRETRAINED=True,
    AMP=True,
    TARGET_FAR=0.05,
    # Models to train per split. Keys are defined in MODEL_SPECS below.
    MODELS=["agef_sharp", "avg_joint", "single_mobilenet", "single_resnet18", "single_effnet_b0"],
    DEEP_ENSEMBLE=True,                        # built post hoc from the three single_* models
    TRIAGE_AXES=("msp", "entropy"),            # the paper's two axes
    KNN_K=50,
    KNN_MAX_TRAIN=6000,
    OPENMAX_TAIL=20,
    OPENMAX_ALPHA=10,
    RUN_DYNAMIC=True,
    DYNAMIC_EXIT_QUANTILES=[0.0, 0.25, 0.5, 0.75, 0.9],
    N_BOOT=10000,
    OUT_DIR="/kaggle/working/auditx_v2",
    # Duplicate-aware splitting (protocol v3): byte-identical or pixel-identical images form one group; a group is never
    # split across train/val/calib/test; groups containing more than one family (label conflicts) are removed; classes that
    # cannot be split (one group > 90 % of the class, or < 10 images left) are excluded from the experiment.
    # Near-identical images (perceptual hashes) are NOT grouped: low-information byte plots share identical dHashes and
    # transitive matching merges whole families; they are reported as a sensitivity analysis (notebook 12b).
    GROUPING=True,
    GROUP_KEYS=(0, 1),            # 0 = file MD5, 1 = pixel MD5, 2 = 256-bit dHash
    UNSPLITTABLE_SHARE=0.9,
    MIN_CLASS_IMAGES=10,
    # Near-duplicate grouping (protocol v4, applied to the datasets listed): within each family, images whose 256-bit dHash
    # differs by <= NEAR_DUP_BITS are merged, closest pairs first, and a merge is refused if the group would exceed
    # NEAR_DUP_CAP images, which prevents transitive chaining across whole families.
    # cap 250 = smallest cap that brings residual near-identical matches (256-bit dHash within 8 bits) between training and
    # evaluation images below 5 % on every dataset without making a class unsplittable through chaining; chosen from duplicate
    # statistics only, never from detection results.
    NEAR_DUP_DATASETS=("Malimg", "MaleVis", "Dumpware10"),
    # Amendment 1 (MaleVis only): identical 256-bit-hash clusters are merged exactly and never cut by the cap.
    IDENTICAL_WHOLE_DATASETS=("MaleVis",),
    NEAR_DUP_BITS=8,
    NEAR_DUP_CAP=250,
)
PROTOCOL = "grouped-L1-v3"

EXPERT_NAMES = ("mobilenet", "resnet18", "effnet_b0")
MODEL_SPECS = {
    # name                 experts            fusion  temp  lambda_div
    "agef_sharp":        (EXPERT_NAMES,       "gate", 1.0,  0.05),
    "agef_soft":         (EXPERT_NAMES,       "gate", 2.0,  0.05),
    "agef_nodiv":        (EXPERT_NAMES,       "gate", 1.0,  0.0),
    "avg_joint":         (EXPERT_NAMES,       "avg",  1.0,  0.0),
    "single_mobilenet":  (("mobilenet",),     "avg",  1.0,  0.0),
    "single_resnet18":   (("resnet18",),      "avg",  1.0,  0.0),
    "single_effnet_b0":  (("effnet_b0",),     "avg",  1.0,  0.0),
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


# -----------------------------------------------------------------------------
# 1. Data
# -----------------------------------------------------------------------------
def make_transform(img):
    return transforms.Compose([
        transforms.Resize((img, img)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


class IndexedSubset(Dataset):
    """Subset of an ImageFolder with labels remapped to 0..K-1 (unknown -> -1)."""
    def __init__(self, base, indices, remap):
        self.base, self.indices, self.remap = base, np.asarray(indices), remap

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        path, t = self.base.samples[self.indices[i]]
        img = self.base.loader(path)
        if self.base.transform is not None:
            img = self.base.transform(img)
        return img, self.remap.get(t, -1)


def _strat_split(idx, y, test_size, seed):
    try:
        return train_test_split(idx, test_size=test_size, stratify=y, random_state=seed)
    except ValueError:  # a class too small to stratify
        return train_test_split(idx, test_size=test_size, random_state=seed)


def image_keys(path):
    """file MD5, decoded grey-level pixel MD5, and 256-bit dHash (17x16) as hex."""
    import hashlib
    from PIL import Image
    b = open(path, "rb").read()
    with Image.open(path) as im:
        g = im.convert("L")
        pmd5 = hashlib.md5(g.tobytes() + str(g.size).encode()).hexdigest()
        a = np.asarray(g.resize((17, 16), Image.BILINEAR), dtype=np.int16)
    return hashlib.md5(b).hexdigest(), pmd5, np.packbits((a[:, 1:] > a[:, :-1]).flatten()).tobytes().hex()


def compute_groups(base, cache_path=None, key_types=(0, 1), unsplittable_share=0.9, min_class_images=10,
                   near_bits=None, near_cap=50):
    """Union of exact-key equalities (+ optional capped within-family near-duplicate merging) -> duplicate groups.
    Returns dict(groups, drop, exclude_classes, stats, protocol)."""
    keys = None
    if cache_path and os.path.exists(cache_path):
        c = json.load(open(cache_path))
        if c.get("n") == len(base.samples): keys = c["keys"]
    if keys is None:
        keys = []
        for i, (p, _) in enumerate(base.samples):
            keys.append(image_keys(p))
            if (i + 1) % 5000 == 0: print(f"    hashed {i + 1}/{len(base.samples)}", flush=True)
        if cache_path:
            os.makedirs(os.path.dirname(cache_path), exist_ok=True)
            json.dump(dict(n=len(base.samples), keys=keys), open(cache_path, "w"))
    n = len(keys); parent = np.arange(n)
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for t in key_types:
        first = {}
        for i, k in enumerate(keys):
            j = first.setdefault(k[t], i)
            if j != i:
                ri, rj = find(i), find(j)
                if ri != rj: parent[ri] = rj
    targets = np.array([t for _, t in base.samples])
    n_near_merges = 0
    if near_bits is not None:
        POP = np.array([bin(i).count("1") for i in range(256)], np.uint8)
        hb = np.stack([np.frombuffer(bytes.fromhex(k[2]), dtype=np.uint8) for k in keys])
        size = np.zeros(n, int)
        for i in range(n): size[find(i)] += 1
        edges = []
        for c in np.unique(targets):
            idx = np.where(targets == c)[0]
            for a0 in range(0, len(idx), 256):
                q = idx[a0:a0 + 256]
                d = POP[hb[q][:, None, :] ^ hb[idx][None, :, :]].sum(-1)
                ii, jj = np.where(d <= near_bits)
                for u, v in zip(ii, jj):
                    if q[u] < idx[v]: edges.append((int(d[u, v]), int(q[u]), int(idx[v])))
        edges.sort()
        for _, i, j in edges:
            ri, rj = find(i), find(j)
            if ri != rj and size[ri] + size[rj] <= near_cap:
                parent[ri] = rj; size[rj] += size[ri]; n_near_merges += 1
    roots = np.array([find(i) for i in range(n)])
    _, groups = np.unique(roots, return_inverse=True)
    fam_per_group = pd.Series(targets).groupby(groups).nunique()
    conflict = fam_per_group[fam_per_group > 1].index.values
    drop = np.isin(groups, conflict)
    sizes = np.bincount(groups)
    exclude = []
    for c in range(len(base.classes)):
        m = (targets == c) & ~drop
        if m.sum() < min_class_images or (m.any() and np.bincount(groups[m]).max() > unsplittable_share * m.sum()):
            exclude.append(c)
    stats = dict(n_images=n, n_groups=int(len(sizes)), n_images_in_multi_groups=int((sizes[groups] > 1).sum()),
                 largest_group=int(sizes.max()), n_conflict_groups=int(len(conflict)), n_dropped=int(drop.sum()),
                 dropped_by_class={base.classes[c]: int(k) for c, k in zip(*np.unique(targets[drop], return_counts=True))},
                 excluded_classes=[base.classes[c] for c in exclude],
                 largest_group_share_by_class={base.classes[c]: round(float(np.bincount(groups[(targets == c) & ~drop]).max()
                                               / max(((targets == c) & ~drop).sum(), 1)), 3) if ((targets == c) & ~drop).any() else None
                                               for c in range(len(base.classes))},
                 near_dup=None if near_bits is None else dict(bits=near_bits, cap=near_cap, merges=n_near_merges))
    prot = PROTOCOL + ("+identical-hash-whole" if 2 in key_types else "") + \
           ("" if near_bits is None else f"+near-L3-{near_bits}bit-cap{near_cap}")
    return dict(groups=groups, drop=drop, exclude_classes=exclude, stats=stats, protocol=prot)


def make_split(base, seed, cfg, G=None):
    """Class-disjoint known/unknown split + 4-way split of known data (70/10/10/10).
    With G (from compute_groups): duplicate groups are kept within one part and label-conflict groups are removed."""
    classes = base.classes
    targets = np.array([t for _, t in base.samples])
    never = {n.lower() for n in cfg["NEVER_UNKNOWN"]}
    eligible = [c for c in range(len(classes)) if classes[c].lower() not in never]
    protected = [classes[c] for c in range(len(classes)) if classes[c].lower() in never]
    if protected:
        print(f"  classes always kept known (benign/other): {protected}")
    n_unk = max(1, int(round(len(classes) * cfg["UNKNOWN_RATIO"])))
    rng = np.random.RandomState(seed)
    unk = sorted(rng.choice(eligible, n_unk, replace=False).tolist())
    known = [c for c in range(len(classes)) if c not in unk]
    excluded = [] if G is None else list(G.get("exclude_classes", []))
    if excluded:
        unk = [c for c in unk if c not in excluded]; known = [c for c in known if c not in excluded]
    keep = np.ones(len(targets), bool) if G is None else ~G["drop"]
    known_idx = np.where(np.isin(targets, known) & keep)[0]
    unk_idx = np.where(np.isin(targets, unk) & keep)[0]
    if G is None:
        f_tr, f_va, f_ca, f_te = cfg["FRACS"]
        tr, rest = _strat_split(known_idx, targets[known_idx], 1 - f_tr, seed)
        va, rest2 = _strat_split(rest, targets[rest], (f_ca + f_te) / (f_va + f_ca + f_te), seed)
        ca, te = _strat_split(rest2, targets[rest2], f_te / (f_ca + f_te), seed)
    else:
        from sklearn.model_selection import StratifiedGroupKFold
        fold = np.full(len(known_idx), -1)
        sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)
        for f, (_, te_) in enumerate(sgkf.split(known_idx, targets[known_idx], G["groups"][known_idx])):
            fold[te_] = f
        tr, va, ca, te = (known_idx[fold <= 6], known_idx[fold == 7], known_idx[fold == 8], known_idx[fold == 9])
        missing = [classes[c] for c in known if not np.isin(c, targets[tr])]
        assert not missing, f"known classes without training images after grouping: {missing}"
        for nm, part in [("val", va), ("calib", ca), ("test", te)]:
            lack = [classes[c] for c in known if not np.isin(c, targets[part])]
            if lack: print(f"  warning: seed {seed}: no {nm} images for {lack} (group sizes)")
    remap = {c: i for i, c in enumerate(known)}
    out = dict(known=known, unknown=unk, remap=remap,
               known_names=[classes[c] for c in known], unknown_names=[classes[c] for c in unk],
               train=tr, val=va, calib=ca, test=te, unk=unk_idx,
               protocol=G.get("protocol", PROTOCOL) if G is not None else "random-v2")
    if G is not None:
        out["n_dropped_conflict"] = int(G["drop"].sum())
        out["excluded_classes"] = [classes[c] for c in excluded]
    return out


def loader(base, idx, remap, cfg, shuffle):
    return DataLoader(IndexedSubset(base, idx, remap), batch_size=cfg["BATCH"], shuffle=shuffle,
                      num_workers=cfg["WORKERS"], pin_memory=torch.cuda.is_available(),
                      persistent_workers=False)


# -----------------------------------------------------------------------------
# 2. Models
# -----------------------------------------------------------------------------
class Expert(nn.Module):
    """Backbone returning penultimate features + its own linear head."""
    def __init__(self, name, num_classes, pretrained):
        super().__init__()
        w = "IMAGENET1K_V1" if pretrained else None
        if name == "mobilenet":
            m = models.mobilenet_v3_small(weights=w); d = m.classifier[3].in_features; m.classifier[3] = nn.Identity()
        elif name == "resnet18":
            m = models.resnet18(weights=w); d = m.fc.in_features; m.fc = nn.Identity()
        elif name == "effnet_b0":
            m = models.efficientnet_b0(weights=w); d = m.classifier[1].in_features; m.classifier[1] = nn.Identity()
        else:
            raise ValueError(name)
        self.name, self.backbone, self.head, self.feat_dim = name, m, nn.Linear(d, num_classes), d

    def forward(self, x):
        f = self.backbone(x)
        return f, self.head(f)


class MultiExpert(nn.Module):
    """AGEF-Net (fusion='gate') or uniform average (fusion='avg'). One expert = single CNN."""
    def __init__(self, expert_names, num_classes, fusion="gate", temp=1.0, pretrained=True):
        super().__init__()
        self.experts = nn.ModuleList([Expert(n, num_classes, pretrained) for n in expert_names])
        self.E, self.fusion, self.temp = len(expert_names), fusion, temp
        if fusion == "gate":
            self.gate = nn.Sequential(nn.Linear(num_classes * self.E, 128), nn.ReLU(), nn.Linear(128, self.E))

    def forward(self, x):
        feats, logits = zip(*[e(x) for e in self.experts])
        stacked = torch.stack(logits, 1)                       # B,E,C
        if self.fusion == "gate":
            w = F.softmax(self.gate(stacked.flatten(1)) / self.temp, dim=1)
        else:
            w = torch.full((x.size(0), self.E), 1.0 / self.E, device=x.device)
        fused = (w.unsqueeze(-1) * stacked).sum(1)
        # per-expert L2 normalisation so no single backbone dominates KNN distances
        return dict(logits=fused, expert_logits=stacked, weights=w,
                    feats=torch.cat([F.normalize(f.float(), dim=1) for f in feats], 1))


# -----------------------------------------------------------------------------
# 3. Training (model selection on known-class val accuracy only)
# -----------------------------------------------------------------------------
def train(model, tr_ld, va_ld, cfg, lambda_div, log_prefix=""):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["LR"], weight_decay=cfg["WD"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["EPOCHS"])
    use_amp = cfg["AMP"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best, best_state, hist = -1.0, None, []
    for ep in range(cfg["EPOCHS"]):
        model.train(); t0 = time.time(); tl, n = 0.0, 0
        for xb, yb in tr_ld:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                out = model(xb)
                loss = F.cross_entropy(out["logits"], yb)
                if lambda_div > 0 and model.E > 1 and model.fusion == "gate":
                    wbar = out["weights"].float().mean(0)
                    loss = loss + lambda_div * (1.0 - (-(wbar * torch.log(wbar + 1e-8)).sum()))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * xb.size(0); n += xb.size(0)
        sched.step()
        acc = evaluate_acc(model, va_ld, use_amp)
        hist.append(dict(epoch=ep, train_loss=tl / n, val_acc=acc, sec=time.time() - t0))
        print(f"{log_prefix} ep {ep:2d} | loss {tl / n:.4f} | val acc {acc:.2f}% | {time.time() - t0:.0f}s", flush=True)
        if acc > best:
            best, best_state = acc, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, best, hist


@torch.no_grad()
def evaluate_acc(model, ld, use_amp=False):
    model.eval(); c = t = 0
    for xb, yb in ld:
        with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
            p = model(xb.to(DEVICE))["logits"].argmax(1).cpu()
        c += (p == yb).sum().item(); t += yb.numel()
    return 100.0 * c / max(t, 1)


@torch.no_grad()
def collect(model, ld):
    """Return float64 numpy outputs for every sample in the loader (fp32 forward)."""
    model.eval(); acc = {k: [] for k in ("logits", "expert_logits", "weights", "feats", "y")}
    for xb, yb in ld:
        out = model(xb.to(DEVICE))
        for k in ("logits", "expert_logits", "weights", "feats"):
            acc[k].append(out[k].float().cpu().numpy().astype(np.float64))
        acc["y"].append(yb.numpy())
    return {k: np.concatenate(v) for k, v in acc.items()}


# -----------------------------------------------------------------------------
# 4. Scores (higher = more likely unknown)
# -----------------------------------------------------------------------------
def entropy(p):
    return -(p * np.log(np.clip(p, 1e-300, 1.0))).sum(-1)


def base_scores(o, is_deep_ensemble=False):
    """o: dict with logits [N,C], expert_logits [N,E,C], weights [N,E]."""
    if is_deep_ensemble:
        pe = softmax(o["expert_logits"], axis=-1)              # N,E,C
        p = pe.mean(1)
        logits = o["expert_logits"].mean(1)
    else:
        logits = o["logits"]; p = softmax(logits, axis=-1)
        pe = softmax(o["expert_logits"], axis=-1)
    s = OrderedDict()
    s["msp"] = -p.max(1)
    s["entropy"] = entropy(p)
    s["maxlogit"] = -logits.max(1)
    s["energy"] = -logsumexp(logits, axis=1)
    if pe.shape[1] > 1:
        s["mutual_info"] = entropy(pe.mean(1)) - entropy(pe).mean(1)
    if (not is_deep_ensemble) and o["weights"].shape[1] > 1 and np.ptp(o["weights"]) > 1e-9:
        s["gate_entropy"] = entropy(o["weights"])
    return s


class KNNScorer:
    """Sun et al. (2022): negative cosine similarity to k-th nearest training feature."""
    def __init__(self, train_feats, k, max_train, seed):
        rng = np.random.RandomState(seed)
        if len(train_feats) > max_train:
            train_feats = train_feats[rng.choice(len(train_feats), max_train, replace=False)]
        self.bank = torch.tensor(train_feats / np.linalg.norm(train_feats, axis=1, keepdims=True), dtype=torch.float32)
        self.k = min(k, len(self.bank))

    def __call__(self, feats):
        q = torch.tensor(feats / np.linalg.norm(feats, axis=1, keepdims=True), dtype=torch.float32)
        out = []
        for i in range(0, len(q), 1024):
            sims = q[i:i + 1024] @ self.bank.T
            out.append(-sims.topk(self.k, dim=1).values[:, -1])
        return torch.cat(out).numpy().astype(np.float64)


class OpenMax:
    """Bendale & Boult (2016) on fused logits: per-class MAV, Weibull fit on the
    largest distances of correctly classified training samples, top-alpha revision."""
    def __init__(self, train_logits, train_y, tail, alpha):
        from scipy.stats import weibull_min
        self.C = train_logits.shape[1]; self.alpha = min(alpha, self.C)
        self.mav = np.zeros((self.C, self.C)); self.params = []
        pred = train_logits.argmax(1)
        for c in range(self.C):
            av = train_logits[(train_y == c) & (pred == c)]
            if len(av) == 0:
                av = train_logits[train_y == c]
            self.mav[c] = av.mean(0)
            d = np.linalg.norm(av - self.mav[c], axis=1)
            t = np.sort(d)[-min(tail, len(d)):]
            if len(t) < 3 or t.std() < 1e-9:
                self.params.append(None)
            else:
                self.params.append(weibull_min.fit(t, floc=0))
        self._cdf = weibull_min.cdf

    def __call__(self, logits):
        N = len(logits); out = np.zeros(N)
        ranks = np.argsort(-logits, axis=1)[:, :self.alpha]
        wts = np.array([(self.alpha - i) / self.alpha for i in range(self.alpha)])
        for n in range(N):
            v = logits[n].copy(); unk = 0.0
            for r, c in enumerate(ranks[n]):
                prm = self.params[c]
                cdf = 0.0 if prm is None else self._cdf(np.linalg.norm(logits[n] - self.mav[c]), *prm)
                om = wts[r] * cdf
                unk += v[c] * om
                v[c] = v[c] * (1 - om)
            z = np.concatenate([[unk], v])
            out[n] = softmax(z)[0]                               # P(unknown)
        return out


# -----------------------------------------------------------------------------
# 5. Metrics
# -----------------------------------------------------------------------------
def fpr_at_95tpr(s_known, s_unk):
    """Known = positive (accepted if score <= thr). FPR = unknowns accepted."""
    thr = np.quantile(s_known, 0.95)
    return float((s_unk <= thr).mean())


def operating_point(s_calib, s_test, s_unk, target_far):
    thr = np.quantile(s_calib, 1 - target_far)
    return dict(
        thr=float(thr),
        zdr=float((s_unk > thr).mean()),
        far_test=float((s_test > thr).mean()),
        far_calib=float((s_calib > thr).mean()),
        tie_frac_calib=float(np.isclose(s_calib, thr, rtol=0, atol=1e-12).mean()),
    )


def threshold_free(s_known_test, s_unk):
    y = np.r_[np.zeros(len(s_known_test)), np.ones(len(s_unk))]
    s = np.r_[s_known_test, s_unk]
    return dict(auroc=float(roc_auc_score(y, s)), aupr_out=float(average_precision_score(y, s)),
                fpr95=fpr_at_95tpr(s_known_test, s_unk))


def four_tag(conf_score, unc_score, tau_c, tau_u):
    """conf_score / unc_score are 'higher = more unknown'. Paper's Eq. 7 mapping:
    0 Auto-block (confident & certain), 1 Quarantine (unconfident & uncertain),
    2 Review (confident & uncertain), 3 Log (unconfident & certain)."""
    lowconf, highunc = conf_score > tau_c, unc_score > tau_u
    return np.where(~lowconf & ~highunc, 0, np.where(lowconf & highunc, 1, np.where(~lowconf & highunc, 2, 3)))


def triage_report(sc_calib, sc_test, sc_unk, axes, target_far):
    a, b = axes
    tau_a = np.quantile(sc_calib[a], 1 - target_far)
    tau_b = np.quantile(sc_calib[b], 1 - target_far)
    tk = four_tag(sc_test[a], sc_test[b], tau_a, tau_b)
    tu = four_tag(sc_unk[a], sc_unk[b], tau_a, tau_b)
    r = dict(tau_a=float(tau_a), tau_b=float(tau_b),
             zdr=float(np.isin(tu, [1, 2]).mean()), far_test=float(np.isin(tk, [1, 2]).mean()),
             wr_test=float(np.isin(tk, [0, 3]).mean()))
    for t, nm in enumerate(["autoblock", "quarantine", "review", "log"]):
        r[f"known_{nm}"] = float((tk == t).mean()); r[f"unk_{nm}"] = float((tu == t).mean())
    return r


# -----------------------------------------------------------------------------
# 6. Dynamic early exit (honest version)
# -----------------------------------------------------------------------------
def measure_expert_costs(model, cfg):
    """FLOPs per image (torch flop counter) and mean latency per 32-image batch, per expert + gate."""
    from torch.utils.flop_counter import FlopCounterMode
    model.eval(); x1 = torch.randn(1, 3, cfg["IMG"], cfg["IMG"], device=DEVICE)
    xb = torch.randn(cfg["BATCH"], 3, cfg["IMG"], cfg["IMG"], device=DEVICE)
    costs = {}
    with torch.no_grad():
        for e in model.experts:
            fc = FlopCounterMode(display=False)
            with fc:
                e(x1)
            for _ in range(3):
                e(xb)
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(10):
                e(xb)
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            costs[e.name] = dict(gflops=fc.get_total_flops() / 1e9, ms_per_batch=(time.time() - t0) / 10 * 1e3)
    return costs


def dynamic_eval(o_cal, o_test, o_unk, costs, cfg):
    """Exit after MobileNet (expert 0) when its own MSP >= theta; theta chosen as a
    quantile of calib MobileNet-MSP so that a target fraction of known traffic exits.
    The detection score is entropy of whichever output is used; its threshold is
    re-calibrated on calib under the same policy."""
    rows = []
    f_m, f_rest = costs["mobilenet"]["gflops"], sum(v["gflops"] for k, v in costs.items() if k != "mobilenet")
    t_m, t_rest = costs["mobilenet"]["ms_per_batch"], sum(v["ms_per_batch"] for k, v in costs.items() if k != "mobilenet")

    def policy(o, theta):
        pm = softmax(o["expert_logits"][:, 0], axis=-1)
        exit_ = pm.max(1) >= theta
        p = np.where(exit_[:, None], pm, softmax(o["logits"], axis=-1))
        return entropy(p), exit_

    mcal = softmax(o_cal["expert_logits"][:, 0], axis=-1).max(1)
    for q in cfg["DYNAMIC_EXIT_QUANTILES"]:
        theta = np.inf if q == 0 else np.quantile(mcal, 1 - q)
        s_cal, _ = policy(o_cal, theta)
        s_te, ex_te = policy(o_test, theta)
        s_un, ex_un = policy(o_unk, theta)
        op = operating_point(s_cal, s_te, s_un, cfg["TARGET_FAR"])
        exit_rate = np.r_[ex_te, ex_un].mean()
        rows.append(dict(target_exit=q, theta=float(theta), exit_rate_known=float(ex_te.mean()),
                         exit_rate_unknown=float(ex_un.mean()),
                         gflops_per_img=f_m + (1 - exit_rate) * f_rest,
                         gflops_static=f_m + f_rest,
                         est_ms_per_batch=t_m + (1 - exit_rate) * t_rest,
                         ms_static=t_m + t_rest, **op))
    return rows


# -----------------------------------------------------------------------------
# 7. One (dataset, seed) run
# -----------------------------------------------------------------------------
def run_split(ds_name, base, seed, cfg):
    out_dir = os.path.join(cfg["OUT_DIR"], ds_name, f"seed{seed}")
    os.makedirs(out_dir, exist_ok=True)
    sp = make_split(base, seed, cfg, cfg.get("_GROUPS", {}).get(ds_name))
    json.dump({k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in sp.items() if k != "remap"},
              open(os.path.join(out_dir, "split.json"), "w"), indent=1)
    print(f"\n[{ds_name} seed {seed}] known={len(sp['known'])} unknown={sp['unknown_names']} "
          f"| train {len(sp['train'])} val {len(sp['val'])} calib {len(sp['calib'])} test {len(sp['test'])} unk {len(sp['unk'])}")

    lds = {k: loader(base, sp[k], sp["remap"], cfg, shuffle=(k == "train")) for k in ("train", "val")}
    eval_lds = {k: loader(base, sp[k], sp["remap"], cfg, shuffle=False) for k in ("train", "calib", "test", "unk")}
    K = len(sp["known"])
    outputs = {}

    for mname in cfg["MODELS"]:
        npz = os.path.join(out_dir, f"{mname}.npz")
        if os.path.exists(npz):
            d = np.load(npz, allow_pickle=True)
            outputs[mname] = {part: {k: d[f"{part}__{k}"] for k in ("logits", "expert_logits", "weights", "feats", "y")}
                              for part in ("train", "calib", "test", "unk")}
            outputs[mname]["_meta"] = json.loads(str(d["meta"]))
            print(f"  [{mname}] loaded cached outputs"); continue
        experts, fusion, temp, lam = MODEL_SPECS[mname]
        seed_everything(seed * 1000 + zlib.crc32(mname.encode()) % 997)
        model = MultiExpert(experts, K, fusion, temp, cfg["PRETRAINED"]).to(DEVICE)
        model, best_val, hist = train(model, lds["train"], lds["val"], cfg, lam, log_prefix=f"  [{mname}]")
        o = {part: collect(model, ld) for part, ld in eval_lds.items()}
        meta = dict(best_val_acc=best_val, history=hist,
                    test_acc=float(100 * (o["test"]["logits"].argmax(1) == o["test"]["y"]).mean()))
        if cfg["RUN_DYNAMIC"] and fusion == "gate":
            meta["costs"] = measure_expert_costs(model, cfg)
        np.savez_compressed(npz, meta=json.dumps(meta),
                            **{f"{p}__{k}": v for p, d in o.items() for k, v in d.items()})
        o["_meta"] = meta; outputs[mname] = o
        del model
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

    singles = [f"single_{n}" for n in EXPERT_NAMES]
    if cfg["DEEP_ENSEMBLE"] and all(s in outputs for s in singles):
        de = {}
        for part in ("train", "calib", "test", "unk"):
            de[part] = dict(
                expert_logits=np.concatenate([outputs[s][part]["expert_logits"] for s in singles], 1),
                feats=np.concatenate([outputs[s][part]["feats"] for s in singles], 1),
                y=outputs[singles[0]][part]["y"])
            de[part]["logits"] = de[part]["expert_logits"].mean(1)
            de[part]["weights"] = np.full((len(de[part]["y"]), 3), 1 / 3)
        p = softmax(de["test"]["expert_logits"], -1).mean(1)
        de["_meta"] = dict(test_acc=float(100 * (p.argmax(1) == de["test"]["y"]).mean()), deep_ensemble=True)
        outputs["deep_ensemble"] = de

    # ---- score + metric every model -------------------------------------------------
    rows, tri_rows, dyn_rows, cls_rows = [], [], [], []
    targets = np.array([t for _, t in base.samples])
    unk_family = np.array([base.classes[t] for t in targets[sp["unk"]]])       # loader order == sp["unk"] order
    for mname, o in outputs.items():
        is_de = mname == "deep_ensemble"
        sc = {part: base_scores(o[part], is_de) for part in ("calib", "test", "unk")}
        knn = KNNScorer(o["train"]["feats"], cfg["KNN_K"], cfg["KNN_MAX_TRAIN"], seed)
        for part in ("calib", "test", "unk"):
            sc[part]["knn"] = knn(o[part]["feats"])
        om = OpenMax(o["train"]["logits"], o["train"]["y"], cfg["OPENMAX_TAIL"], cfg["OPENMAX_ALPHA"])
        for part in ("calib", "test", "unk"):
            sc[part]["openmax"] = om(o[part]["logits"])
        for sname in sc["test"]:
            r = dict(dataset=ds_name, seed=seed, model=mname, score=sname,
                     test_acc=o["_meta"]["test_acc"], n_unknown_classes=len(sp["unknown"]))
            r.update(threshold_free(sc["test"][sname], sc["unk"][sname]))
            r.update(operating_point(sc["calib"][sname], sc["test"][sname], sc["unk"][sname], cfg["TARGET_FAR"]))
            rows.append(r)
        # per-class alarm rate: recall for each held-out family, false-alarm rate for each known class
        test_cls = np.array([sp["known_names"][y] for y in o["test"]["y"]])
        for sname in ("msp", "entropy", "energy", "knn"):
            if sname not in sc["test"]:
                continue
            thr = np.quantile(sc["calib"][sname], 1 - cfg["TARGET_FAR"])
            for fam in np.unique(unk_family):
                m_ = unk_family == fam
                cls_rows.append(dict(dataset=ds_name, seed=seed, model=mname, score=sname, cls=fam, role="unknown",
                                     n=int(m_.sum()), alarm_rate=float((sc["unk"][sname][m_] > thr).mean())))
            for kc in np.unique(test_cls):
                m_ = test_cls == kc
                cls_rows.append(dict(dataset=ds_name, seed=seed, model=mname, score=sname, cls=kc, role="known",
                                     n=int(m_.sum()), alarm_rate=float((sc["test"][sname][m_] > thr).mean())))
        tri = triage_report(sc["calib"], sc["test"], sc["unk"], cfg["TRIAGE_AXES"], cfg["TARGET_FAR"])
        tri_rows.append(dict(dataset=ds_name, seed=seed, model=mname, axes="/".join(cfg["TRIAGE_AXES"]), **tri))
        if cfg["RUN_DYNAMIC"] and "costs" in o["_meta"]:
            for d in dynamic_eval(o["calib"], o["test"], o["unk"], o["_meta"]["costs"], cfg):
                dyn_rows.append(dict(dataset=ds_name, seed=seed, model=mname, **d))
    return rows, tri_rows, dyn_rows, cls_rows


# -----------------------------------------------------------------------------
# 8. Aggregation across splits
# -----------------------------------------------------------------------------
def boot_ci(x, n_boot, seed=0):
    x = np.asarray(x, float)
    if len(x) < 2:
        return (np.nan, np.nan)
    rng = np.random.RandomState(seed)
    m = rng.choice(x, (n_boot, len(x)), replace=True).mean(1)
    return tuple(np.percentile(m, [2.5, 97.5]))


def summarise(raw, cfg):
    metrics = ["auroc", "aupr_out", "fpr95", "zdr", "far_test", "tie_frac_calib", "test_acc"]
    out = []
    for (d, m, s), g in raw.groupby(["dataset", "model", "score"]):
        r = dict(dataset=d, model=m, score=s, n_splits=len(g))
        for k in metrics:
            r[f"{k}_mean"], r[f"{k}_std"] = g[k].mean(), g[k].std(ddof=1) if len(g) > 1 else np.nan
        r["auroc_ci_lo"], r["auroc_ci_hi"] = boot_ci(g["auroc"], cfg["N_BOOT"])
        r["zdr_ci_lo"], r["zdr_ci_hi"] = boot_ci(g["zdr"], cfg["N_BOOT"])
        out.append(r)
    return pd.DataFrame(out)


def paired_tests(raw, cfg, ref=("agef_sharp", "entropy")):
    """Reference (model, score) vs every other (model, score), paired over split seeds."""
    out = []
    for d, gd in raw.groupby("dataset"):
        a = gd[(gd.model == ref[0]) & (gd.score == ref[1])].set_index("seed")
        if a.empty:
            continue
        for (m, s), gb in gd.groupby(["model", "score"]):
            if (m, s) == ref:
                continue
            b = gb.set_index("seed")
            common = a.index.intersection(b.index)
            if len(common) < 2:
                continue
            for k in ("auroc", "zdr"):
                diff = a.loc[common, k].values - b.loc[common, k].values
                t = stats.ttest_rel(a.loc[common, k], b.loc[common, k])
                try:
                    w = stats.wilcoxon(diff).pvalue if np.any(diff != 0) else 1.0
                except ValueError:
                    w = np.nan
                lo, hi = boot_ci(diff, cfg["N_BOOT"])
                out.append(dict(dataset=d, reference=f"{ref[0]}:{ref[1]}", other=f"{m}:{s}", metric=k,
                                n=len(common), mean_diff=diff.mean(), diff_ci_lo=lo, diff_ci_hi=hi,
                                t_p=t.pvalue, wilcoxon_p=w))
    return pd.DataFrame(out)


# -----------------------------------------------------------------------------
# 9. Smoke test data (synthetic, CPU) — verifies the pipeline end to end
# -----------------------------------------------------------------------------
def build_synthetic(root, n_classes=6, n_per=40, img=32, seed=0):
    from PIL import Image
    rng = np.random.RandomState(seed)
    for c in range(n_classes):
        os.makedirs(os.path.join(root, f"fam{c}"), exist_ok=True)
        base = rng.rand(img, img)
        for i in range(n_per):
            a = np.clip(base + 0.25 * rng.randn(img, img), 0, 1)
            Image.fromarray((a * 255).astype(np.uint8), "L").save(os.path.join(root, f"fam{c}", f"{i}.png"))
    os.makedirs(os.path.join(root, "Other"), exist_ok=True)
    for i in range(n_per):
        Image.fromarray((rng.rand(img, img) * 255).astype(np.uint8), "L").save(os.path.join(root, "Other", f"{i}.png"))


# -----------------------------------------------------------------------------
# 10. Main
# -----------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--datasets", nargs="*", help="subset of CFG['DATASETS'] keys")
    ap.add_argument("--path", nargs="*", help="override paths as Name=/path")
    ap.add_argument("--seeds", nargs="*", type=int)
    ap.add_argument("--models", nargs="*")
    ap.add_argument("--epochs", type=int)
    ap.add_argument("--img", type=int)
    ap.add_argument("--batch", type=int)
    ap.add_argument("--workers", type=int)
    ap.add_argument("--out")
    ap.add_argument("--no-pretrained", action="store_true")
    ap.add_argument("--no-dynamic", action="store_true")
    ap.add_argument("--no-grouping", action="store_true", help="old random splits (protocol v2)")
    ap.add_argument("--smoke", action="store_true", help="synthetic CPU run to verify the pipeline")
    ap.add_argument("--resume-from", help="folder with a previous OUT_DIR (e.g. an attached Kaggle output); cached runs are copied and skipped")
    ap.add_argument("--max-hours", type=float, default=None, help="do not start a new split after this many hours")
    args, _ = ap.parse_known_args()

    cfg = copy.deepcopy(CFG)
    if args.smoke:
        root = "/tmp/auditx_smoke_data"
        if not os.path.exists(root):
            build_synthetic(root)
        cfg.update(DATASETS=OrderedDict(Synthetic=root), NEVER_UNKNOWN=["other"], SPLIT_SEEDS=[0, 1], EPOCHS=2, IMG=64, BATCH=16,
                   WORKERS=0, PRETRAINED=False, OUT_DIR="/tmp/auditx_smoke_out", N_BOOT=500,
                   KNN_K=5, OPENMAX_TAIL=5, UNKNOWN_RATIO=0.34)
    if args.path:
        for kv in args.path:
            k, v = kv.split("=", 1); cfg["DATASETS"][k] = v
    if args.datasets:
        cfg["DATASETS"] = OrderedDict((k, cfg["DATASETS"][k]) for k in args.datasets)
    if args.seeds: cfg["SPLIT_SEEDS"] = args.seeds
    if args.models: cfg["MODELS"] = args.models
    if args.epochs: cfg["EPOCHS"] = args.epochs
    if args.img: cfg["IMG"] = args.img
    if args.batch: cfg["BATCH"] = args.batch
    if args.workers is not None: cfg["WORKERS"] = args.workers
    if args.out: cfg["OUT_DIR"] = args.out
    if args.no_pretrained: cfg["PRETRAINED"] = False
    if args.no_dynamic: cfg["RUN_DYNAMIC"] = False
    if args.no_grouping: cfg["GROUPING"] = False

    os.makedirs(cfg["OUT_DIR"], exist_ok=True)
    if args.resume_from and os.path.isdir(args.resume_from):
        import shutil
        n_copied = 0
        for dp, _, fns in os.walk(args.resume_from):
            for fn in fns:
                if fn.endswith(".npz") or fn == "split.json":
                    rel = os.path.relpath(os.path.join(dp, fn), args.resume_from)
                    dst = os.path.join(cfg["OUT_DIR"], rel)
                    if not os.path.exists(dst):
                        os.makedirs(os.path.dirname(dst), exist_ok=True)
                        shutil.copy2(os.path.join(dp, fn), dst); n_copied += 1
        print(f"Resumed {n_copied} cached files from {args.resume_from}")
    t_start = time.time()
    json.dump({k: (list(v.items()) if isinstance(v, OrderedDict) else v) for k, v in cfg.items()},
              open(os.path.join(cfg["OUT_DIR"], "config.json"), "w"), indent=1)
    print(f"Device: {DEVICE} | torch {torch.__version__} | out: {cfg['OUT_DIR']}")

    all_rows, all_tri, all_dyn, all_cls = [], [], [], []
    for ds_name, path in cfg["DATASETS"].items():
        if not path or not os.path.isdir(path):
            print(f"!! skipping {ds_name}: path not found ({path})"); continue
        base = datasets.ImageFolder(path, transform=make_transform(cfg["IMG"]))
        print(f"\n===== {ds_name}: {len(base)} images, {len(base.classes)} classes | protocol {PROTOCOL if cfg['GROUPING'] else 'random-v2'} =====")
        if cfg["GROUPING"]:
            near = ds_name in cfg.get("NEAR_DUP_DATASETS", ())
            G = compute_groups(base, os.path.join(cfg["OUT_DIR"], ds_name, "image_keys.json"), cfg["GROUP_KEYS"],
                               cfg["UNSPLITTABLE_SHARE"], cfg["MIN_CLASS_IMAGES"],
                               cfg["NEAR_DUP_BITS"] if near else None, cfg["NEAR_DUP_CAP"])
            print("  split protocol:", G["protocol"])
            cfg.setdefault("_GROUPS", {})[ds_name] = G
            json.dump(G["stats"], open(os.path.join(cfg["OUT_DIR"], ds_name, "groups_stats.json"), "w"), indent=1)
            print("  duplicate groups:", json.dumps({k: v for k, v in G["stats"].items() if k not in ("dropped_by_class", "largest_group_share_by_class")}),
                  "| dropped (label conflicts):", G["stats"]["dropped_by_class"], "| excluded classes:", G["stats"]["excluded_classes"])
        for seed in cfg["SPLIT_SEEDS"]:
            if args.max_hours and (time.time() - t_start) / 3600 > args.max_hours:
                print(f"!! time budget of {args.max_hours} h reached; not starting seed {seed}. "
                      f"Re-run with --resume-from to continue."); break
            r, t, d, c = run_split(ds_name, base, seed, cfg)
            all_cls += c
            pd.DataFrame(all_cls).to_csv(os.path.join(cfg["OUT_DIR"], "per_class_raw.csv"), index=False)
            print(f"  elapsed {(time.time() - t_start) / 3600:.2f} h", flush=True)
            all_rows += r; all_tri += t; all_dyn += d
            # write after every split so a timed-out session still leaves results
            pd.DataFrame(all_rows).to_csv(os.path.join(cfg["OUT_DIR"], "results_raw.csv"), index=False)
            pd.DataFrame(all_tri).to_csv(os.path.join(cfg["OUT_DIR"], "triage_raw.csv"), index=False)
            if all_dyn:
                pd.DataFrame(all_dyn).to_csv(os.path.join(cfg["OUT_DIR"], "dynamic_raw.csv"), index=False)

    if not all_rows:
        print("No results."); return
    raw = pd.DataFrame(all_rows)
    summ = summarise(raw, cfg); summ.to_csv(os.path.join(cfg["OUT_DIR"], "summary.csv"), index=False)
    paired_tests(raw, cfg).to_csv(os.path.join(cfg["OUT_DIR"], "paired_tests.csv"), index=False)
    tri = pd.DataFrame(all_tri)
    tri.drop(columns=["axes", "seed"]).groupby(["dataset", "model"]).agg(["mean", "std"]).to_csv(os.path.join(cfg["OUT_DIR"], "triage_summary.csv"))
    if all_dyn:
        dyn = pd.DataFrame(all_dyn)
        dyn.groupby(["dataset", "model", "target_exit"]).mean(numeric_only=True).to_csv(
            os.path.join(cfg["OUT_DIR"], "dynamic_summary.csv"))

    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
    show = summ[summ.score.isin(["msp", "entropy", "energy", "knn", "openmax"])][
        ["dataset", "model", "score", "n_splits", "auroc_mean", "auroc_std", "fpr95_mean",
         "zdr_mean", "zdr_std", "far_test_mean", "tie_frac_calib_mean", "test_acc_mean"]]
    print("\n", show.round(4).to_string(index=False))
    print(f"\nAll outputs in {cfg['OUT_DIR']}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/auditx_analysis.py
# =============================================================================
# AUDIT-X FINAL ANALYSIS - every table, figure and verdict of the paper, from the three final runs (PROTOCOL.md)
# Inputs (anywhere under ROOT): auditx_final_malimg/, auditx_final_malevis/, auditx_final_dumpware10/ (complete MANIFEST),
#   optionally xrep_raw.csv (nb 08), fewshot_raw.csv (nb 10), fewshot_baselines_raw.csv (nb 14).
# =============================================================================
import glob, json, os, sys, time, zlib
import numpy as np, pandas as pd
from scipy import stats
from scipy.special import softmax, logsumexp
from scipy.stats import rankdata, beta
import auditx_eval_v2 as ax

ROOT = os.environ.get("AUDITX_ROOT", "/kaggle/input")
OUT = os.environ.get("AUDITX_OUT", "/kaggle/working/auditx_final_analysis")
N_BOOT = int(os.environ.get("AUDITX_NBOOT", 1000))
TOST_MARGIN, ALPHA, FAR5, FAR1, MIN_EXC, N_REDRAW = 0.03, 0.05, 0.05, 0.01, 10, 200
KNN_KS = [1, 5, 10, 50]
DATASETS = ["Malimg", "MaleVis", "Dumpware10"]
MODELS = ["agef_sharp", "agef_tuned", "avg_joint", "single_mobilenet", "single_resnet18", "single_effnet_b0"]
SINGLES = ["single_mobilenet", "single_resnet18", "single_effnet_b0"]
PROTOCOL = "grouped-L1-v3+near-L3-8bit-cap250"
MALEVIS_VERSION = os.environ.get("AUDITX_MALEVIS", "v5")   # "v5": amended MaleVis run (primary); "v4": original run (sensitivity)
EXPECTED = {"Malimg": PROTOCOL, "Dumpware10": PROTOCOL, "MaleVis": "grouped-L1-v3+identical-hash-whole+near-L3-8bit-cap250" if MALEVIS_VERSION == "v5" else PROTOCOL}
DEDUP_EVAL = os.environ.get("AUDITX_DEDUP_EVAL", "0") == "1"   # deviation-driven sensitivity analysis (not pre-registered)
DEDUP_BITS = 8
os.makedirs(OUT, exist_ok=True)
LOG = []
def say(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); LOG.append(s)

# ----------------------------------------------------------------------------- locate runs
RUNS = {}
for ds in DATASETS:
    name = f"auditx_final_{ds.lower()}" + ("_v5" if ds == "MaleVis" and MALEVIS_VERSION == "v5" else "")
    hits = [os.path.dirname(p) for p in glob.glob(f"{ROOT}/**/{name}/MANIFEST.json", recursive=True)]
    if not hits: say(f"!! {ds}: no final run found"); continue
    d = sorted(hits)[0]; man = json.load(open(f"{d}/MANIFEST.json"))
    assert man["complete"], f"{ds}: run incomplete ({man['n_done']}/{man['n_expected']}) - finish it before analysing"
    assert man["protocol"] == EXPECTED[ds], f"{ds}: protocol {man['protocol']} != {EXPECTED[ds]}"
    RUNS[ds] = dict(dir=d, man=man, targets=np.load(f"{d}/targets.npy"), classes=json.load(open(f"{d}/classes.json")),
                    splits=sorted(int(p.split("split")[-1]) for p in glob.glob(f"{d}/split*") if os.path.isdir(p)),
                    seeds=man["args"]["train_seeds"])
    say(f"{ds}: {d} | splits {RUNS[ds]['splits']} | seeds {RUNS[ds]['seeds']} | protocol OK | script {man['script_sha256'][:12]}"
        f" | protocol file {str(man.get('protocol_sha256'))[:12]}")
assert RUNS, "no runs found"
_ph = {ds: R["man"].get("protocol_sha256") for ds, R in RUNS.items()}
_sh = {ds: R["man"]["script_sha256"] for ds, R in RUNS.items()}
say("protocol-document sha256 per run:", _ph)
say("training-script sha256 per run:", _sh)
_orig = [ds for ds in RUNS if EXPECTED[ds] == PROTOCOL]
if None in _ph.values() or len({_ph[ds] for ds in _orig}) > 1 or len({_sh[ds] for ds in _orig}) > 1:
    say("!! WARNING: runs under the original protocol do not share the same protocol document and script - report as a deviation")
else:
    say("runs under the original protocol share one protocol document and one script" +
        ("; MaleVis uses Amendment 1 (different document and script hash, as intended)" if "MaleVis" in RUNS and EXPECTED["MaleVis"] != PROTOCOL else ""))

# ----------------------------------------------------------------------------- deduplicated evaluation masks (optional)
POP8 = np.array([bin(i).count("1") for i in range(256)], np.uint8)
MASK, MASK_STATS = {}, []
if DEDUP_EVAL:
    for ds, R in RUNS.items():
        keys = json.load(open(f"{R['dir']}/image_keys.json"))["keys"]
        f0 = np.array([k[0] for k in keys]); f1 = np.array([k[1] for k in keys])
        hb = np.stack([np.frombuffer(bytes.fromhex(k[2]), dtype=np.uint8) for k in keys])
        for s in R["splits"]:
            sp = json.load(open(f"{R['dir']}/split{s}/split.json")); tr = np.array(sp["train"])
            for p in ("calib", "test", "unk"):
                q = np.array(sp[p])
                dmin = np.concatenate([POP8[hb[q[i:i + 256]][:, None] ^ hb[tr][None]].sum(-1).min(1) for i in range(0, len(q), 256)])
                keep = (dmin > DEDUP_BITS) & ~np.isin(f0[q], f0[tr]) & ~np.isin(f1[q], f1[tr])
                MASK[(ds, s, p)] = keep
                MASK_STATS.append(dict(dataset=ds, split=s, part=p, n=len(q), removed=int((~keep).sum()), removed_share=float((~keep).mean())))
    pd.DataFrame(MASK_STATS).to_csv(f"{OUT}/T_dedup_removed.csv", index=False)
    say("DEDUPLICATED EVALUATION: evaluation images with a training image within", DEDUP_BITS, "bits (or identical file/pixels) are removed")
    say(pd.DataFrame(MASK_STATS).groupby(["dataset", "part"])[["n", "removed"]].sum().to_string())

def _apply_mask(ds, s, entry):
    if not DEDUP_EVAL: return entry
    removed_any = any((~MASK[(ds, s, p)]).any() for p in ("calib", "test", "unk"))
    if removed_any and min(MASK[(ds, s, p)].sum() for p in ("calib", "test", "unk")) < 10:
        return None      # too few evaluation images left in this split: the split is skipped (reported in the log)
    for p in ("calib", "test", "unk"):
        k = MASK[(ds, s, p)]
        for sn in entry["sc"][p]: entry["sc"][p][sn] = entry["sc"][p][sn][k]
        if entry.get("pred") is not None: entry["pred"][p] = entry["pred"][p][k]
        if entry.get("y") is not None and p in entry["y"]: entry["y"][p] = entry["y"][p][k]
        if p == "unk": entry["unk_fam"] = entry["unk_fam"][k]
        if entry.get("ee") is not None:
            entry["ee"]["m0"][p] = entry["ee"]["m0"][p][k]; entry["ee"]["fused"][p] = entry["ee"]["fused"][p][k]
    return entry

# ----------------------------------------------------------------------------- scoring helpers
def maha(train_f, train_y, K, dim=512):
    from sklearn.covariance import LedoitWolf
    from sklearn.decomposition import PCA
    pca = PCA(n_components=min(dim, train_f.shape[1], len(train_f) - 1), random_state=0).fit(train_f)
    z = pca.transform(train_f); mu = np.stack([z[train_y == c].mean(0) for c in range(K)])
    cov = LedoitWolf().fit(z - mu[train_y]).covariance_; ev, V = np.linalg.eigh(cov); W = V / np.sqrt(np.clip(ev, 1e-12, None)); mw = mu @ W
    return lambda f: (((pca.transform(f) @ W) ** 2).sum(1)[:, None] - 2 * (pca.transform(f) @ W) @ mw.T + (mw ** 2).sum(1)[None]).min(1)

def vim(raw_tr, W, b, raw, logit):
    D = W.shape[1]; dim = 1000 if D >= 2048 else (512 if D >= 768 else D // 2)
    u = -np.linalg.pinv(W) @ b; X = raw_tr.astype(np.float64) - u
    ev, V = np.linalg.eigh(X.T @ X / len(X)); NS = V[:, :D - dim]
    alpha = (raw_tr.astype(np.float64) @ W.T + b).max(1).mean() / np.linalg.norm(X @ NS, axis=1).mean()
    return alpha * np.linalg.norm((raw.astype(np.float64) - u) @ NS, axis=1) - logsumexp(logit, axis=1)

def credibility(cal_s, cal_pred, s, pred):
    """Transcend-style: class-conditional conformal p-value of a nonconformity score (higher = stranger); returns 1 - p."""
    out = np.empty(len(s))
    for c in np.unique(pred):
        ref = np.sort(cal_s[cal_pred == c]) if (cal_pred == c).sum() >= 5 else np.sort(cal_s)
        m = pred == c; out[m] = (len(ref) - np.searchsorted(ref, s[m], side="left") + 1) / (len(ref) + 1)
    return 1 - out

def auroc(k, u):
    r = rankdata(np.r_[k, u]); return (r[len(k):].sum() - len(u) * (len(u) + 1) / 2) / (len(u) * len(k))

def cthr(x, a):
    n = len(x); k = int(np.ceil((n + 1) * (1 - a))); return np.inf if k > n else np.sort(x)[k - 1]

def load_npz(p):
    z = np.load(p, allow_pickle=True); parts = {}
    for k in z.files:
        if "__" in k:
            pp, kk = k.split("__", 1); parts.setdefault(pp, {})[kk] = z[k].astype(np.float64) if kk not in ("y",) else z[k]
    return parts, json.loads(str(z["meta"])), ({"W": z["W"].astype(np.float64), "b": z["b"].astype(np.float64)} if "W" in z.files else None)

# ----------------------------------------------------------------------------- score everything once
SC = {}   # (ds, split, seed, model) -> dict
t0 = time.time()
for ds, R in RUNS.items():
    for s in R["splits"]:
        sp = json.load(open(f"{R['dir']}/split{s}/split.json")); K = len(sp["known"])
        unk_fam = np.array([R["classes"][t] for t in R["targets"][sp["unk"]]])
        dfe = np.load(f"{R['dir']}/dinov2_feats.npy").astype(np.float64) if os.path.exists(f"{R['dir']}/dinov2_feats.npy") else None
        for t in R["seeds"]:
            outs, metas, heads = {}, {}, {}
            for m in MODELS:
                outs[m], metas[m], heads[m] = load_npz(f"{R['dir']}/split{s}/seed{t}/{m}.npz")
            de = {}
            for p in ("train", "calib", "test", "unk"):
                de[p] = dict(expert_logits=np.concatenate([outs[x][p]["expert_logits"] for x in SINGLES], 1),
                             feats=np.concatenate([outs[x][p]["feats"] for x in SINGLES], 1), y=outs[SINGLES[0]][p]["y"],
                             react_logits=np.mean([outs[x][p]["react_logits"] for x in SINGLES], 0))
                de[p]["logits"] = de[p]["expert_logits"].mean(1); de[p]["weights"] = np.full((len(de[p]["y"]), 3), 1 / 3)
            outs["deep_ensemble"] = de
            for m, o in outs.items():
                is_de = m == "deep_ensemble"
                sc = {p: ax.base_scores(o[p], is_de) for p in ("train", "calib", "test", "unk")}
                for k in KNN_KS:
                    kn = ax.KNNScorer(o["train"]["feats"], k, 6000, s)
                    for p in ("calib", "test", "unk"): sc[p][f"knn{k}"] = kn(o[p]["feats"])
                mh = maha(o["train"]["feats"], o["train"]["y"], K)
                for p in ("calib", "test", "unk"):
                    sc[p]["mahalanobis"] = mh(o[p]["feats"])
                    rl = o[p]["react_logits"]; sc[p]["react_energy"] = -logsumexp(rl, axis=1)
                if m.startswith("single_") or is_de:
                    mem = [m] if not is_de else SINGLES
                    for p in ("calib", "test", "unk"):
                        sc[p]["vim"] = np.mean([vim(outs[x]["train"]["raw"], heads[x]["W"], heads[x]["b"], outs[x][p]["raw"],
                                                    outs[x][p]["logits"]) for x in mem], 0)
                if t == R["seeds"][0]:
                    om = ax.OpenMax(o["train"]["logits"], o["train"]["y"], 20, 10)
                    for p in ("calib", "test", "unk"): sc[p]["openmax"] = om(o[p]["logits"])
                pr = {p: (softmax(o[p]["expert_logits"], -1).mean(1) if is_de else softmax(o[p]["logits"], -1)).argmax(1) for p in ("calib", "test", "unk")}
                for p in ("calib", "test", "unk"):
                    sc[p]["transcend"] = credibility(sc["calib"]["knn50"], pr["calib"], sc[p]["knn50"], pr[p])
                val_acc = max(h["val_acc"] for h in metas[m]["history"]) if m in metas else \
                          np.mean([max(h["val_acc"] for h in metas[x]["history"]) for x in SINGLES])
                SC[(ds, s, t, m)] = dict(sc=sc, pred=pr, y={p: o[p]["y"] for p in ("calib", "test")}, unk_fam=unk_fam.copy(), val_acc=val_acc,
                                         hist=metas.get(m, {}).get("history"),
                                         ee=dict(m0={p: o[p]["expert_logits"][:, 0] for p in ("calib", "test", "unk")},
                                                 fused={p: o[p]["logits"] for p in ("calib", "test", "unk")}) if m == "agef_sharp" and t == R["seeds"][0] else None)
                SC[(ds, s, t, m)] = _apply_mask(ds, s, SC[(ds, s, t, m)])
                if SC[(ds, s, t, m)] is None:
                    del SC[(ds, s, t, m)]
                    if m == MODELS[0] and t == R["seeds"][0]: say(f"!! {ds} split {s}: too few evaluation images after deduplication - split skipped")
            if dfe is not None and t == R["seeds"][0]:
                tr = np.array(sp["train"]); remap = {c: i for i, c in enumerate(sp["known"])}; tr_y = np.array([remap[c] for c in R["targets"][tr]])
                fe = dfe / np.linalg.norm(dfe, axis=1, keepdims=True)
                sub = np.random.RandomState(s).choice(len(tr), min(len(tr), 6000), replace=False)
                kn = ax.KNNScorer(fe[tr][sub], 50, 6000, s); mh = maha(fe[tr][sub], tr_y[sub], K)
                sc = {p: {"knn50": kn(fe[np.array(sp[p])]), "mahalanobis": mh(fe[np.array(sp[p])])} for p in ("calib", "test", "unk")}
                SC[(ds, s, t, "dinov2_frozen")] = _apply_mask(ds, s, dict(sc=sc, pred=None, y=None, unk_fam=unk_fam.copy(), val_acc=np.nan, hist=None, ee=None))
                if SC[(ds, s, t, "dinov2_frozen")] is None: del SC[(ds, s, t, "dinov2_frozen")]
            del outs
        say(f"{ds} split {s}: scored ({(time.time() - t0) / 60:.1f} min)")
SCORES_ALL = sorted({sname for d in SC.values() for sname in d["sc"]["test"]})

# ----------------------------------------------------------------------------- long table of per-run metrics
rows = []
for (ds, s, t, m), d in SC.items():
    for sn in d["sc"]["test"]:
        c, te, u = d["sc"]["calib"][sn], d["sc"]["test"][sn], d["sc"]["unk"][sn]
        th5, th1 = cthr(c, FAR5), cthr(c, FAR1)
        y = np.r_[np.zeros(len(te)), np.ones(len(u))]; z = np.r_[te, u]
        rows.append(dict(dataset=ds, split=s, seed=t, model=m, score=sn, auroc=auroc(te, u),
                         aupr=float(__import__("sklearn.metrics", fromlist=["average_precision_score"]).average_precision_score(y, z)),
                         fpr95=float((u <= np.quantile(te, 0.95)).mean()), zdr5=float((u > th5).mean()), far5=float((te > th5).mean()),
                         zdr1=float((u > th1).mean()), far1=float((te > th1).mean()), n_calib=len(c),
                         acc=float((d["pred"]["test"] == d["y"]["test"]).mean()) if d["pred"] is not None else np.nan, val_acc=d["val_acc"]))
L = pd.DataFrame(rows); L.to_csv(f"{OUT}/runs_long.csv", index=False)
MAIN = L.groupby(["dataset", "model", "score"]).agg(auroc=("auroc", "mean"), auroc_sd=("auroc", "std"), aupr=("aupr", "mean"), fpr95=("fpr95", "mean"),
        zdr5=("zdr5", "mean"), far5=("far5", "mean"), zdr1=("zdr1", "mean"), far1=("far1", "mean"), acc=("acc", "mean"), n=("auroc", "size")).reset_index()
MAIN.to_csv(f"{OUT}/T_main_all_scores.csv", index=False)
say("main table written:", len(MAIN), "rows")

# ----------------------------------------------------------------------------- confirmatory comparisons (family cluster bootstrap)
def runs_of(ds, cfg):
    return {(s, t): SC[(ds, s, t, cfg[0])] for (d_, s, t, m) in SC if d_ == ds and m == cfg[0]}
def best_single(ds, s, t):
    return max(SINGLES, key=lambda x: SC[(ds, s, t, x)]["val_acc"])
def boot(ds, a, b, n_boot, seed=0):
    keys = sorted({(s, t) for (d_, s, t, m) in SC if d_ == ds and m == "agef_sharp"})
    def get(cfg, s, t):
        m = best_single(ds, s, t) if cfg[0] == "best_single" else cfg[0]; return SC[(ds, s, t, m)]["sc"]
    fams = sorted({f for (s, t) in keys for f in SC[(ds, s, t, "agef_sharp")]["unk_fam"]})
    pt = np.mean([auroc(get(a, s, t)["test"][a[1]], get(a, s, t)["unk"][a[1]]) - auroc(get(b, s, t)["test"][b[1]], get(b, s, t)["unk"][b[1]]) for s, t in keys])
    rng = np.random.RandomState(seed); bs = []
    for _ in range(n_boot):
        fb = rng.choice(fams, len(fams)); diffs = []
        for s, t in keys:
            uf = SC[(ds, s, t, "agef_sharp")]["unk_fam"]
            idx = np.concatenate([np.where(uf == f)[0] for f in fb if (uf == f).any()] or [np.array([], int)])
            if not len(idx): continue
            A, B = get(a, s, t), get(b, s, t); kb = rng.randint(0, len(A["test"][a[1]]), len(A["test"][a[1]]))
            diffs.append(auroc(A["test"][a[1]][kb], A["unk"][a[1]][idx]) - auroc(B["test"][b[1]][kb], B["unk"][b[1]][idx]))
        if diffs: bs.append(np.mean(diffs))
    return pt, np.array(bs)
CONF = [("C1", ("deep_ensemble", "energy"), ("agef_sharp", "energy")), ("C2", ("deep_ensemble", "knn50"), ("agef_sharp", "knn50")),
        ("C3", ("avg_joint", "energy"), ("agef_sharp", "energy")), ("C4", ("agef_tuned", "energy"), ("agef_sharp", "energy")),
        ("C5", ("agef_tuned", "energy"), ("deep_ensemble", "energy")), ("C6", ("deep_ensemble", "energy"), ("best_single", "energy"))]
crow = []
for cid, a, b in CONF:
    per = {ds: boot(ds, a, b, N_BOOT, seed=zlib.crc32(cid.encode()) % 1000) for ds in RUNS
           if any(k[0] == ds and k[3] == "agef_sharp" for k in SC)}
    per = {ds: v for ds, v in per.items() if len(v[1])}
    if not per: say(f"{cid}: no evaluable splits - skipped"); continue
    nmin = min(len(v[1]) for v in per.values()); pooled_bs = np.mean([v[1][:nmin] for v in per.values()], 0)
    pooled_pt = np.mean([v[0] for v in per.values()])
    for ds, (pt, bs) in list(per.items()) + [("pooled", (pooled_pt, pooled_bs))]:
        crow.append(dict(id=cid, config=f"{a[0]}+{a[1]}", reference=f"{b[0]}+{b[1]}", dataset=ds, delta=pt,
                         ci95_lo=np.percentile(bs, 2.5), ci95_hi=np.percentile(bs, 97.5), ci90_lo=np.percentile(bs, 5), ci90_hi=np.percentile(bs, 95),
                         p=min(1.0, 2 * min((bs <= 0).mean(), (bs >= 0).mean()))))
    say(f"{cid} done")
C = pd.DataFrame(crow); pooled = C[C.dataset == "pooled"].copy()
order = pooled.p.sort_values().index; run = 0
for r, i in enumerate(order):
    run = max(run, min(1.0, (len(order) - r) * pooled.loc[i, "p"])); C.loc[i, "p_holm"] = run
C["tost_equivalent"] = (C.ci90_lo > -TOST_MARGIN) & (C.ci90_hi < TOST_MARGIN)
C.to_csv(f"{OUT}/T_confirmatory.csv", index=False)

# ----------------------------------------------------------------------------- variance decomposition
vrow = []
for ds in sorted({k[0] for k in SC}):
    for m, sn in [("agef_sharp", "energy"), ("agef_tuned", "energy"), ("avg_joint", "energy"), ("deep_ensemble", "energy"), ("deep_ensemble", "knn50"), ("single_mobilenet", "energy")]:
        g = L[(L.dataset == ds) & (L.model == m) & (L.score == sn)].pivot_table(index="split", columns="seed", values="auroc")
        T = g.shape[1]; w = g.var(axis=1, ddof=1).mean(); bt = max(g.mean(axis=1).var(ddof=1) - w / T, 0)
        vrow.append(dict(dataset=ds, config=f"{m}+{sn}", mean=g.values.mean(), sd_seed=np.sqrt(w), sd_split=np.sqrt(bt), share_seed=w / (w + bt) if w + bt > 0 else np.nan))
V = pd.DataFrame(vrow); V.to_csv(f"{OUT}/T_variance.csv", index=False)

# ----------------------------------------------------------------------------- evaluation shortcuts (relative to the pool mean)
POOL = [(m, sn) for m in MODELS + ["deep_ensemble"] for sn in ["msp", "entropy", "maxlogit", "energy", "knn50", "mahalanobis"]]
srow, scal = [], []; rng = np.random.RandomState(0)
for ds in sorted({k[0] for k in SC}):
    g = L[(L.dataset == ds)]; g = g[g.apply(lambda r: (r.model, r.score) in POOL, axis=1)]
    piv = g.groupby(["model", "score", "split"]).zdr5.mean().unstack("split")
    pm = piv.values.mean(); s3 = piv.max(0).mean(); s4 = piv.loc[piv.mean(1).idxmax()].max()
    ko = []   # known-data-only rule: model with the highest calibration accuracy, energy score
    for s in RUNS[ds]["splits"]:
        for t in RUNS[ds]["seeds"]:
            if (ds, s, t, "agef_sharp") not in SC: continue
            best = max(MODELS + ["deep_ensemble"], key=lambda x: (SC[(ds, s, t, x)]["pred"]["calib"] == SC[(ds, s, t, x)]["y"]["calib"]).mean())
            ko.append(L[(L.dataset == ds) & (L.split == s) & (L.seed == t) & (L.model == best) & (L.score == "energy")].zdr5.iloc[0])
    # S1: thresholds on (a subsample of) training data, FAR reported on training data vs measured on held-out test data
    s1z, s1r, s1t = [], [], []
    for (d_, s, t, m), d in SC.items():
        if d_ != ds or (m, "energy") not in POOL: continue
        for sn in ["msp", "entropy", "maxlogit", "energy"]:
            th = cthr(d["sc"]["train"][sn], FAR5); s1z.append((d["sc"]["unk"][sn] > th).mean())
            s1r.append((d["sc"]["train"][sn] > th).mean()); s1t.append((d["sc"]["test"][sn] > th).mean())
    lf = g[g.score.isin(["msp", "entropy", "maxlogit", "energy"])]
    srow.append(dict(dataset=ds, pool_size=len(piv), pool_mean_zdr=pm, s3_same_split_best=s3, s4_best_config_best_split=s4,
                     known_only_rule_zdr=np.mean(ko), infl_s3_pp=100 * (s3 - pm), infl_s4_pp=100 * (s4 - pm),
                     s1_traincal_zdr=np.mean(s1z), leakfree_logit_zdr=lf.zdr5.mean(), infl_s1_pp=100 * (np.mean(s1z) - lf.zdr5.mean()),
                     s1_far_reported=np.mean(s1r), s1_far_true=np.mean(s1t), far_leakfree=lf.far5.mean()))
    for msize in [1, 2, 4, 8, 16, 32, len(piv)]:
        if msize > len(piv): continue
        v = [(lambda sub: sub.max(0).mean() - sub.values.mean())(piv.iloc[rng.choice(len(piv), msize, replace=False)]) for _ in range(300)]
        scal.append(dict(dataset=ds, pool_size=msize, s3_inflation_pp=100 * np.mean(v)))
S = pd.DataFrame(srow); S.to_csv(f"{OUT}/T_shortcuts.csv", index=False); pd.DataFrame(scal).to_csv(f"{OUT}/T_shortcut_pool_scaling.csv", index=False)

# ----------------------------------------------------------------------------- 1 % operating point
def cp(k, n):
    return (beta.ppf(0.025, k, n - k + 1) if k > 0 else 0.0, beta.ppf(0.975, k + 1, n - k) if k < n else 1.0)
lrow = []
for (ds, s, t, m), d in SC.items():
    if t != RUNS[ds]["seeds"][0]: continue
    for sn in ["energy", "knn50", "mahalanobis"]:
        if sn not in d["sc"]["test"]: continue
        c, te, u = d["sc"]["calib"][sn], d["sc"]["test"][sn], d["sc"]["unk"][sn]; th = cthr(c, FAR1); k = int((te > th).sum())
        pool = np.r_[c, te]; r = np.random.RandomState(s); fr, zr = [], []
        for _ in range(N_REDRAW):
            p = r.permutation(len(pool)); h = len(pool) // 2; th2 = cthr(pool[p[:h]], FAR1)
            fr.append((pool[p[h:]] > th2).mean()); zr.append((u > th2).mean())
        lo, hi = cp(k, len(te))
        lrow.append(dict(dataset=ds, split=s, model=m, score=sn, n_calib=len(c), reliable=len(c) * FAR1 >= MIN_EXC, zdr=(u > th).mean(),
                         far=k / len(te), far_lo=lo, far_hi=hi, redraw_far_sd=np.std(fr), redraw_zdr_mean=np.mean(zr), redraw_zdr_sd=np.std(zr)))
LF = pd.DataFrame(lrow).groupby(["dataset", "model", "score"]).mean(numeric_only=True).reset_index(); LF.to_csv(f"{OUT}/T_lowfar.csv", index=False)

# ----------------------------------------------------------------------------- residual duplicates across partitions
POP = np.array([bin(i).count("1") for i in range(256)], np.uint8); drow = []
for ds, R in RUNS.items():
    keys = json.load(open(f"{R['dir']}/image_keys.json"))["keys"]; f0 = np.array([k[0] for k in keys]); f1 = np.array([k[1] for k in keys])
    hb = np.stack([np.frombuffer(bytes.fromhex(k[2]), dtype=np.uint8) for k in keys])
    for s in R["splits"]:
        if DEDUP_EVAL and (ds, s, 0, "agef_sharp") not in SC and not any(k[:2] == (ds, s) for k in SC): continue
        sp = json.load(open(f"{R['dir']}/split{s}/split.json")); tr = np.array(sp["train"])
        ev = np.r_[np.array(sp["calib"])[MASK[(ds, s, "calib")]], np.array(sp["test"])[MASK[(ds, s, "test")]]] if DEDUP_EVAL else np.r_[sp["calib"], sp["test"]]
        dmin = np.concatenate([POP[hb[ev[i:i + 256]][:, None] ^ hb[tr][None]].sum(-1).min(1) for i in range(0, len(ev), 256)])
        drow.append(dict(dataset=ds, split=s, L0_file=np.isin(f0[ev], f0[tr]).mean(), L1_pixels=np.isin(f1[ev], f1[tr]).mean(),
                         L2_identical_hash=(dmin == 0).mean(), L3_hash_le8=(dmin <= 8).mean()))
DU = pd.DataFrame(drow).groupby("dataset").mean(numeric_only=True).drop(columns="split"); DU.to_csv(f"{OUT}/T_residual_duplicates.csv")

# ----------------------------------------------------------------------------- per-family recall, Mondrian, early exit, triage note
frow = []
for (ds, s, t, m), d in SC.items():
    if m != "deep_ensemble": continue
    th = cthr(d["sc"]["calib"]["energy"], FAR5)
    for f in np.unique(d["unk_fam"]):
        mk = d["unk_fam"] == f; frow.append(dict(dataset=ds, split=s, seed=t, family=f, recall=(d["sc"]["unk"]["energy"][mk] > th).mean()))
FAM = pd.DataFrame(frow).groupby(["dataset", "family"]).recall.agg(["mean", "min", "max", "count"]).reset_index(); FAM.to_csv(f"{OUT}/T_families.csv", index=False)
mrow = []
for (ds, s, t, m), d in SC.items():
    if m != "deep_ensemble" or "Other" not in RUNS[ds]["classes"]: continue
    sp = json.load(open(f"{RUNS[ds]['dir']}/split{s}/split.json")); ob = sp["known_names"].index("Other"); sc = d["sc"]
    g = cthr(sc["calib"]["energy"], FAR5); pc, pt, pu = d["pred"]["calib"], d["pred"]["test"], d["pred"]["unk"]
    thc = {c: (cthr(sc["calib"]["energy"][pc == c], FAR5) if (pc == c).sum() >= 20 else g) for c in range(len(sp["known"]))}
    yt = d["y"]["test"]; ben = yt == ob
    mrow.append(dict(dataset=ds, split=s, seed=t, zdr_global=(sc["unk"]["energy"] > g).mean(), zdr_mondrian=(sc["unk"]["energy"] > np.array([thc[c] for c in pu])).mean(),
                     benign_far_global=(sc["test"]["energy"][ben] > g).mean(), benign_far_mondrian=(sc["test"]["energy"][ben] > np.array([thc[c] for c in pt[ben]])).mean(),
                     unknown_predicted_benign=(pu == ob).mean()))
MO = pd.DataFrame(mrow).groupby("dataset").mean(numeric_only=True).drop(columns=["split", "seed"]) if mrow else pd.DataFrame(); MO.to_csv(f"{OUT}/T_mondrian.csv")
erow = []; fm, fr_ = 0.111, 3.627 + 0.769
for (ds, s, t, m), d in SC.items():
    if d["ee"] is None: continue
    e = d["ee"]; pm = {p: softmax(e["m0"][p], -1) for p in e["m0"]}; mc = pm["calib"].max(1)
    for q in [0, 0.25, 0.5, 0.75, 0.9]:
        th = np.inf if q == 0 else np.quantile(mc, 1 - q)
        def pol(p):
            ex = pm[p].max(1) >= th; pr = np.where(ex[:, None], pm[p], softmax(e["fused"][p], -1)); return ax.entropy(pr), ex
        sc_, _ = pol("calib"); st, ek = pol("test"); su, eu = pol("unk"); th5 = cthr(sc_, FAR5)
        erow.append(dict(dataset=ds, split=s, target_exit=q, exit_known=ek.mean(), exit_unknown=eu.mean(),
                         saving_known=1 - (fm + (1 - ek.mean()) * fr_) / (fm + fr_), saving_unknown=1 - (fm + (1 - eu.mean()) * fr_) / (fm + fr_),
                         zdr=(su > th5).mean(), far=(st > th5).mean()))
EE = pd.DataFrame(erow).groupby(["dataset", "target_exit"]).mean(numeric_only=True).drop(columns="split"); EE.to_csv(f"{OUT}/T_early_exit.csv")
LAT = {ds: json.load(open(f"{R['dir']}/latency.json")) for ds, R in RUNS.items() if os.path.exists(f"{R['dir']}/latency.json")}
json.dump(LAT, open(f"{OUT}/T_latency.json", "w"), indent=1)
trow = []
for (ds, s, t, m), d in SC.items():
    if m not in ("deep_ensemble", "agef_sharp") or t != RUNS[ds]["seeds"][0]: continue
    sc = d["sc"]; ta = cthr(sc["calib"]["msp"], FAR5); tb = cthr(sc["calib"]["entropy"], FAR5)
    A, B = sc["unk"]["msp"] > ta, sc["unk"]["entropy"] > tb
    trow.append(dict(dataset=ds, split=s, model=m, unk_review_or_log=((A ^ B)).mean(), unk_quarantine=(A & B).mean()))
TRI = pd.DataFrame(trow).groupby(["dataset", "model"]).mean(numeric_only=True).drop(columns="split"); TRI.to_csv(f"{OUT}/T_triage_note.csv")

# ----------------------------------------------------------------------------- cross-representation and few-shot (completed runs 08 / 10 / 14)
for name in ["xrep_raw.csv", "fewshot_raw.csv", "fewshot_baselines_raw.csv"]:
    fs = [f for f in glob.glob(f"{ROOT}/**/{name}", recursive=True) if "QUICKTEST" not in f]
    if fs:
        X = pd.read_csv(fs[0]); keys = [c for c in ["direction", "model", "score", "k", "method"] if c in X.columns]
        X.groupby(keys).mean(numeric_only=True).reset_index().to_csv(f"{OUT}/T_{name.replace('_raw.csv', '')}_summary.csv", index=False)
        say(f"{name}: summarised from {fs[0]}")
    else:
        say(f"{name}: not attached (section skipped)")

# ----------------------------------------------------------------------------- verdicts
say("=" * 100)
say("EVALUATION:", "deduplicated (sensitivity analysis, deviation from PROTOCOL.md)" if DEDUP_EVAL else "as pre-registered")
for cid, a, b in CONF:
    if not ((C.id == cid) & (C.dataset == "pooled")).any(): continue
    r = C[(C.id == cid) & (C.dataset == "pooled")].iloc[0]
    per = "; ".join(f"{x.dataset} {x.delta:+.3f} [{x.ci95_lo:+.3f}, {x.ci95_hi:+.3f}]" for x in C[(C.id == cid) & (C.dataset != "pooled")].itertuples())
    verdict = ("SUPERIOR" if r.ci95_lo > 0 and r.p_holm < ALPHA else "INFERIOR" if r.ci95_hi < 0 and r.p_holm < ALPHA else
               "EQUIVALENT (TOST)" if r.tost_equivalent else "NO SIGNIFICANT DIFFERENCE")
    say(f"{cid} {r.config} vs {r.reference}: pooled {r.delta:+.3f} [{r.ci95_lo:+.3f}, {r.ci95_hi:+.3f}], Holm p {r.p_holm:.4f} -> {verdict} | {per}")
say("Residual duplicates across partitions (mean over splits):\n" + DU.round(4).to_string())
say("=" * 100)
open(f"{OUT}/VERDICTS.txt", "w").write("\n".join(LOG))
print("outputs:", sorted(os.listdir(OUT)))

In [ ]:
import subprocess, sys, os
MALEVIS = "v5"   # "v5": FINAL_A2 (primary); "v4": original FINAL_A run (sensitivity)
env = dict(os.environ, AUDITX_MALEVIS=MALEVIS, AUDITX_DEDUP_EVAL="1", AUDITX_OUT="/kaggle/working/auditx_final_analysis_dedup")
p = subprocess.Popen([sys.executable, "/kaggle/working/auditx_analysis.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd="/kaggle/working", env=env)
for line in p.stdout: print(line, end="", flush=True)
p.wait(); assert p.returncode == 0, "analysis failed - send the log above"

In [ ]:
import pandas as pd, os
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
O = "/kaggle/working/auditx_final_analysis_dedup"
print(open(f"{O}/VERDICTS.txt").read()[-6000:])
print("\n### images removed"); display(pd.read_csv(f"{O}/T_dedup_removed.csv").groupby(["dataset", "part"])[["n", "removed"]].sum())
for f in ["T_confirmatory.csv", "T_variance.csv", "T_shortcuts.csv", "T_residual_duplicates.csv", "T_lowfar.csv"]:
    print("\n###", f); display(pd.read_csv(f"{O}/{f}").round(4))
M = pd.read_csv(f"{O}/T_main_all_scores.csv")
display(M.pivot_table(index=["dataset", "model"], columns="score", values="auroc").round(3))